<a href="https://colab.research.google.com/github/mateoneo/CatorDog/blob/main/CatorDogclassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount = True)

Mounted at /content/drive


In [2]:
!cp -r "/content/drive/MyDrive/catdog" /content/data
!ls /content/data

c1.jpg	c3.jpg	c5.jpg	d1.jpg	d3.jpg	d5.jpg
c2.jpg	c4.jpg	c6.jpg	d2.jpg	d4.jpg	d6.jpg


In [3]:
import glob
print(len(glob.glob('/content/data/**/*.jpg', recursive=True)), "jpgs found")

12 jpgs found


In [4]:
import os, shutil, glob
os.makedirs('/content/sorted/cats', exist_ok=True)
os.makedirs('/content/sorted/dogs', exist_ok=True)
for f in glob.glob('/content/data/**/*.jpg', recursive=True):
    name = os.path.basename(f).lower()
    if 'cat' in name:
        shutil.copy(f, '/content/sorted/cats')
    elif 'dog' in name:
        shutil.copy(f, '/content/sorted/dogs')
print(len(os.listdir('/content/sorted/cats')), "cats,", len(os.listdir('/content/sorted/dogs')), "dogs")

0 cats, 0 dogs


In [5]:
import os, shutil, glob
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn.functional as F
import torch.optim as optim

# Correctly sort the files since the filenames are 'c1.jpg', 'd1.jpg', etc.
os.makedirs('/content/sorted/cats', exist_ok=True)
os.makedirs('/content/sorted/dogs', exist_ok=True)

for f in glob.glob('/content/data/**/*.jpg', recursive=True):
    name = os.path.basename(f).lower()
    if name.startswith('c'):
        shutil.copy(f, '/content/sorted/cats')
    elif name.startswith('d'):
        shutil.copy(f, '/content/sorted/dogs')

print(f"Sorted files successfully: {len(os.listdir('/content/sorted/cats'))} cats, {len(os.listdir('/content/sorted/dogs'))} dogs")

# Define transforms and Dataset
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

data = datasets.ImageFolder('/content/sorted', transform=transform)
print("Dataset classes:", data.classes)
trainloader = DataLoader(data, batch_size=32, shuffle=True)
train_size = int(0.8 * len(data))
train_data, val_data = random_split(data, [train_size, len(data) - train_size])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 32, 3, padding=1)
        self.pool = torch.nn.MaxPool2d(2, 2, padding=1)
        self.conv2 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = torch.nn.Conv2d(64, 128, 3, padding=1)
        # For 128x128 input, the shape after 3 poolings becomes 17x17
        self.fc1 = torch.nn.Linear(128 * 17 * 17, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x

net = Net()
criterion = torch.nn.CrossEntropyLoss()
learningrate = 0.001
device = torch.device("cpu")
optimizer = torch.optim.Adam(net.parameters(), lr=learningrate)
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            if outputs.ndim == 1 or outputs.shape[1] == 1:   # one output neuron (sigmoid)
                preds = (torch.sigmoid(outputs.view(-1)) > 0.5).long()
            else:                                            # two output neurons (softmax)
                preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

for epoch in range(30):
  running_loss = 0.0
  if epoch == 15:
    learningrate = 0.001
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
    if i % 10 == 9 or i == len(trainloader) - 1:
      print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / (i + 1)))
      running_loss = 0.0
labels, preds = evaluate(net, val_loader, device)

print("Accuracy:", round(accuracy_score(labels, preds), 4))
print("\nConfusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(labels, preds))
print("\n", classification_report(labels, preds, target_names=["cat", "dog"]))

Sorted files successfully: 6 cats, 6 dogs
Dataset classes: ['cats', 'dogs']
[1,     1] loss: 0.693
[2,     1] loss: 0.857
[3,     1] loss: 0.655
[4,     1] loss: 0.673
[5,     1] loss: 0.652
[6,     1] loss: 0.611
[7,     1] loss: 0.587
[8,     1] loss: 0.532
[9,     1] loss: 0.485
[10,     1] loss: 0.415
[11,     1] loss: 0.375
[12,     1] loss: 0.303
[13,     1] loss: 0.233
[14,     1] loss: 0.200
[15,     1] loss: 0.131
[16,     1] loss: 0.084
[17,     1] loss: 0.061
[18,     1] loss: 0.027
[19,     1] loss: 0.019
[20,     1] loss: 0.009
[21,     1] loss: 0.003
[22,     1] loss: 0.002
[23,     1] loss: 0.002
[24,     1] loss: 0.001
[25,     1] loss: 0.001
[26,     1] loss: 0.000
[27,     1] loss: 0.000
[28,     1] loss: 0.000
[29,     1] loss: 0.000
[30,     1] loss: 0.000
Accuracy: 0.3333

Confusion matrix (rows = actual, cols = predicted):
[[1 1]
 [1 0]]

               precision    recall  f1-score   support

         cat       0.50      0.50      0.50         2
         dog     